<a href="https://colab.research.google.com/github/michalozeryflato/biomed-multi-alignment/blob/main/mammal_inference_wdr91.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAMMAL Model Inference

Run inference with a finetuned MAMMAL model on **any CSV file** containing SMILES strings.

All dataset, dataloader and task logic lives in **`mammal_utils.py`** (same folder).

## Steps
0. Configuration — edit paths here
1. Preview input data
2. Load tokenizer
3. Load model
4. Build DataLoader
5. Run inference
6. Inspect predictions
7. Evaluate (if labels available)
8. Enrichment metrics (Precision@K / EF@K)
9. Top hits

In [ ]:
!pip install biomed-multi-alignment

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 482.8/482.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 1) Clone the specific branch that holds your notebook and utils
!git clone -b mainframe26 https://github.com/BiomedSciAI/biomed-multi-alignment.git

# 2) Move into the folder that contains both the notebook and mammal_utils.py
%cd /content/biomed-multi-alignment/tutorials/MAINFRAME_2026

# 3) Sanity check: should list mammal_utils.py
import os
print(os.listdir('.'))

Cloning into 'biomed-multi-alignment'...
remote: Enumerating objects: 1234, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 1234 (delta 209), reused 161 (delta 119), pack-reused 851 (from 2)
Receiving objects: 100% (1234/1234), 1.91 MiB | 5.94 MiB/s, done.
Resolving deltas: 100% (769/769), done.
/content/biomed-multi-alignment/tutorials/MAINFRAME_2026
['MAMMAL_inference.ipynb', 'mammal_utils.py']


## 0. Configuration

**Edit the paths below before running.**

In [ ]:
import os
import torch

# ── USER SETTINGS ──────────────────────────────────────────────────────────────

# Path to the finetuned Lightning checkpoint (.ckpt)
# MODEL_PATH = "/proj/ligand_ai/models/PGK2_DEL_cdd_to_creative_mammal/last-v2.ckpt"
# MODEL_PATH = "michalozeryflato/biomed.omics.bl.sm.ma-ted-458m.pgk2_del_cdd"
MODEL_PATH = "michalozeryflato/biomed.omics.bl.sm.ma-ted-458m.wdr91_asms"
# Path to the inference CSV/TSV file
# DATA_PATH  = "/proj/ligand_ai/datasets_manager/processed/splits/PGK2_DEL_cdd_to_creative/test.csv"
DATA_PATH = "/content/test.csv"

# Where to write the predictions CSV
# OUTPUT_PATH = "/proj/ligand_ai/results/PGK2_DEL_cdd_to_creative/mammal_notebook_predictions.csv"
OUTPUT_PATH = "/content/results/mammal_notebook_predictions.csv"

# Column names in the input CSV
SMILES_COLUMN = "SMILES"
LABEL_COLUMN  = "label"   # set to None if the file has no labels

# Base MAMMAL tokenizer / model (HuggingFace hub id or local path)
BASE_MODEL_PATH = "ibm/biomed.omics.bl.sm.ma-ted-458m"

# Inference device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# DataLoader settings
BATCH_SIZE            = 128
DRUG_MAX_SEQ_LENGTH   = 300
ENCODER_INPUT_MAX_LEN = 320
LABELS_MAX_LEN        = 10
NUM_WORKERS           = 0

# ── VALIDATION ─────────────────────────────────────────────────────────────────
# assert os.path.exists(MODEL_PATH), f"Model not found: {MODEL_PATH}"
assert os.path.exists(DATA_PATH),  f"Data not found:  {DATA_PATH}"
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

print(f"Model : {MODEL_PATH}")
print(f"Data  : {DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Device: {DEVICE}")

Model : michalozeryflato/biomed.omics.bl.sm.ma-ted-458m.wdr91_asms
Data  : /content/test.csv
Output: /content/results/mammal_notebook_predictions.csv
Device: cuda


## 1. Preview Input Data

In [ ]:
import pandas as pd

sep = "\t" if DATA_PATH.endswith(".tsv") else ","
input_df = pd.read_csv(DATA_PATH, sep=sep)

print(f"Shape  : {input_df.shape}")
print(f"Columns: {list(input_df.columns)}")

if LABEL_COLUMN and LABEL_COLUMN in input_df.columns:
    print(f"\nLabel distribution:")
    print(input_df[LABEL_COLUMN].value_counts().sort_index())

input_df.head()

Shape  : (234, 2)
Columns: ['SMILES', 'Label']


,SMILES,Label
0,COC(=O)CC(NC(=O)c1ccc(Cn2cncn2)cc1)c1ccc(Cl)cc1,1
1,NC(=O)C[C@@H](NC(=O)c1ccc(N2CCC(O)C2)cc1)c1ccc...,1
2,COC(=O)CC(NC(=O)c1ccc(CN2C(=O)CCC2=O)cc1)c1ccc...,1
3,O=C(NC(CO)c1ccc(Cl)cc1)c1ccc(N2CCC(O)C2)cc1,1
4,CN(C)Cc1ccc(C(=O)NC(c2ccccc2)c2ccc(Cl)cc2)cn1,1


## 2. Load Tokenizer

In [ ]:
# !git pull
# import importlib
# import mammal_utils
# importlib.reload(mammal_utils)

In [ ]:
from mammal_utils import load_tokenizer

tokenizer_op = load_tokenizer(BASE_MODEL_PATH)
print("Tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

gene_tokenizer.json: 0.00B [00:00, ?B/s]

cell_attributes_tokenizer.json: 0.00B [00:00, ?B/s]

t5_tokenizer_AA_special.json: 0.00B [00:00, ?B/s]

(…)th_aug_4272372_samples_balanced_1_1.json: 0.00B [00:00, ?B/s]

config.yaml:   0%|          | 0.00/967 [00:00<?, ?B/s]

Tokenizer loaded.


## 3. Load Model

In [ ]:
from mammal_utils import load_model

model = load_model(MODEL_PATH, BASE_MODEL_PATH, device=DEVICE)
print(model)

Path doesn't exist. Will try to download from hf hub. pretrained_model_name_or_path='ibm/biomed.omics.bl.sm.ma-ted-458m'


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

mammal.png:   0%|          | 0.00/353k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Attempting to load model from dir: pretrained_model_name_or_path='/root/.cache/huggingface/hub/models--ibm--biomed.omics.bl.sm.ma-ted-458m/snapshots/6d319d8dcf97f8821635327fc8cda24670553daa'


last.ckpt:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

checkpoint_path='/root/.cache/huggingface/hub/models--michalozeryflato--biomed.omics.bl.sm.ma-ted-458m.wdr91_asms/snapshots/b5dd9ef62a0190dd7c8a7b6579f128561cfa9056/last.ckpt'
Mammal(
  (t5_model): T5ForConditionalGeneration(
    (shared): Embedding(100001, 768)
    (encoder): T5Stack(
      (embed_tokens): Embedding(100001, 768)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
                (relative_attention_bias): Embedding(32, 12)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): 

## 4. Build DataLoader

In [ ]:
from mammal_utils import build_dataloader

dataloader = build_dataloader(
    df=input_df,
    tokenizer_op=tokenizer_op,
    smiles_column=SMILES_COLUMN,
    label_column=LABEL_COLUMN,
    batch_size=BATCH_SIZE,
    drug_max_seq_length=DRUG_MAX_SEQ_LENGTH,
    encoder_input_max_seq_len=ENCODER_INPUT_MAX_LEN,
    labels_max_seq_len=LABELS_MAX_LEN,
    num_workers=NUM_WORKERS,
)

InferenceDataset: 234 samples, has_labels=False
DataLoader ready: 234 samples, 2 batches


## 5. Run Inference

In [ ]:
from mammal_utils import run_inference

predictions_df = run_inference(
    model=model,
    dataloader=dataloader,
    tokenizer_op=tokenizer_op,
    device=DEVICE,
)

predictions_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved {len(predictions_df)} predictions to: {OUTPUT_PATH}")

Inference:   0%|          | 0/2 [00:00<?, ?it/s]


Saved 234 predictions to: /content/results/mammal_notebook_predictions.csv


## 6. Inspect Predictions

In [ ]:
print(f"Shape: {predictions_df.shape}")
print(f"\nPredicted label distribution:")
print(predictions_df["predicted_label"].value_counts().sort_index())
print(f"\nScore statistics:")
print(predictions_df["prediction_score"].describe())
predictions_df.head(10)

Shape: (234, 7)

Predicted label distribution:
predicted_label
0    221
1     13
Name: count, dtype: int64

Score statistics:
count    2.340000e+02
mean     5.189873e-02
std      1.805408e-01
min      2.888450e-08
25%      4.647894e-07
50%      2.303302e-06
75%      2.817561e-04
max      9.343612e-01
Name: prediction_score, dtype: float64


,sample_id,smiles,true_label,predicted_label,prediction_score,raw_score_negative,raw_score_positive
0,0,COC(=O)CC(NC(=O)c1ccc(Cn2cncn2)cc1)c1ccc(Cl)cc1,None,0,0.198176,0.801823,0.198176
1,1,NC(=O)C[C@@H](NC(=O)c1ccc(N2CCC(O)C2)cc1)c1ccc(Cl)cc1,None,1,0.856292,0.143707,0.856292
2,2,COC(=O)CC(NC(=O)c1ccc(CN2C(=O)CCC2=O)cc1)c1ccc(Cl)cc1,None,0,0.019416,0.980580,0.019416
3,3,O=C(NC(CO)c1ccc(Cl)cc1)c1ccc(N2CCC(O)C2)cc1,None,0,0.154179,0.845811,0.154179
4,4,CN(C)Cc1ccc(C(=O)NC(c2ccccc2)c2ccc(Cl)cc2)cn1,None,0,0.000640,0.999358,0.000640
5,5,COCCC(NC(=O)c1ccc(N2CCC(O)C2)cc1)c1ccc(Cl)cc1,None,0,0.001893,0.998107,0.001893
6,6,O=C(O)CC(NC(=O)c1ccc(CNC(=O)c2cccs2)cc1)c1ccc(Cl)cc1,None,0,0.006382,0.993615,0.006382
7,7,COC(=O)CC(NC(=O)c1ccc2c(=O)n3c(nc2c1)CCC3)c1ccc(Cl)cc1,None,0,0.013369,0.986621,0.013369
8,8,COC(=O)CC(NC(=O)c1ccc2c(c1)CCN2C(C)=O)c1ccc(Cl)cc1,None,0,0.365900,0.634098,0.365900
9,9,Cn1ccnc1C(NC(=O)c1ccc(CNC(N)=O)cc1)c1ccc(Cl)cc1,None,1,0.870945,0.129054,0.870945


## 7. Evaluate (if labels available)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

has_labels = predictions_df["true_label"].notna().any()

if has_labels:
    valid   = predictions_df[predictions_df["true_label"].notna()].copy()
    y_true  = valid["true_label"].astype(int)
    y_pred  = valid["predicted_label"].astype(int)
    y_score = valid["prediction_score"].astype(float)

    print("=" * 50)
    print("CLASSIFICATION METRICS")
    print("=" * 50)
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1 Score  : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC AUC   : {roc_auc_score(y_true, y_score):.4f}")
    print(f"PR AUC    : {average_precision_score(y_true, y_score):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(ax=ax, colorbar=False)
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("No true labels found — skipping evaluation.")

No true labels found — skipping evaluation.


## 8. Enrichment Metrics

Precision@K and Enrichment Factor (EF@K).

**EF@K** = Precision@K / hit_prevalence  
where hit_prevalence = fraction of positives in the full dataset.

In [ ]:
import numpy as np
from mammal_utils import calculate_enrichment_metrics

if has_labels:
    n_total = len(valid)
    top_k_values = sorted({k for k in [10, 50, 100, 500, 1000, 2000] if k <= n_total})
    if n_total not in top_k_values:
        top_k_values.append(n_total)

    enrichment = calculate_enrichment_metrics(
        y_true.values, y_score.values, top_k_values=top_k_values
    )

    n_positives = int(y_true.sum())
    hit_prev    = n_positives / n_total

    print("=" * 60)
    print("ENRICHMENT METRICS")
    print("=" * 60)
    print(f"Dataset : {n_total} compounds  |  Positives: {n_positives}  |  Hit rate: {hit_prev:.2%}")
    print()
    print(f"{'K':>8}  {'Precision@K':>12}  {'EF@K':>8}  {'Hits in top-K':>14}")
    print("-" * 50)
    for k in top_k_values:
        prec = enrichment.get(f"Precision@{k}", np.nan)
        ef   = enrichment.get(f"EF@{k}",        np.nan)
        hits = int(round(prec * k)) if not np.isnan(prec) else "N/A"
        print(f"{k:>8}  {prec:>12.4f}  {ef:>8.2f}  {hits:>14}")
    print("=" * 60)

    ef_rows = [
        {
            "K":             k,
            "Precision@K":   enrichment.get(f"Precision@{k}", np.nan),
            "EF@K":          enrichment.get(f"EF@{k}",        np.nan),
            "Hits_in_top_K": int(round(enrichment.get(f"Precision@{k}", 0) * k)),
        }
        for k in top_k_values
    ]
    ef_df = pd.DataFrame(ef_rows)
    ef_df
else:
    print("No true labels found — skipping enrichment metrics.")

No true labels found — skipping enrichment metrics.


## 9. Top Hits

In [ ]:
top_hits = (
    predictions_df[predictions_df["predicted_label"] == 1]
    .sort_values("prediction_score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

n_pos = (predictions_df["predicted_label"] == 1).sum()
print(f"Top 20 predicted positives (out of {n_pos} total predicted positives):")
top_hits[["sample_id", "smiles", "prediction_score", "true_label"]]

Top 20 predicted positives (out of 13 total predicted positives):


,sample_id,smiles,prediction_score,true_label
0,37,CCn1c(=O)[nH]c2cc(C(=O)NC(c3ccc(F)cc3)c3cnn(C)c3)ccc2c1=O,0.934361,None
1,182,CC(C)(C)OC(=O)N1CCn2nc(C(=O)NCc3ccc(Cl)c4cccnc34)cc2C1,0.919713,None
2,39,COC(=O)CC(NC(=O)c1cc2c(C)nn(C)c2s1)c1ccc(Cl)cc1,0.908779,None
3,9,Cn1ccnc1C(NC(=O)c1ccc(CNC(N)=O)cc1)c1ccc(Cl)cc1,0.870945,None
4,1,NC(=O)C[C@@H](NC(=O)c1ccc(N2CCC(O)C2)cc1)c1ccc(Cl)cc1,0.856292,None
5,22,Cn1ccnc1C(NC(=O)c1ccc2c(c1)CCCN2S(C)(=O)=O)c1ccc(Cl)cc1,0.792626,None
6,40,Cn1ccnc1C(NC(=O)c1ccc2c(c1)NC(=O)CO2)c1ccc(Cl)cc1,0.746857,None
7,26,O=C(O)CC(NC(=O)c1ccc(CN2CCCC2=O)cc1)c1ccc(Cl)cc1,0.735491,None
8,34,C[C@@H](NC(=O)c1ccc2c(c1)NC(=O)CO2)c1cccc(Cl)c1,0.700300,None
9,35,COC(=O)CC(NC(=O)c1ccc2c(c1)OCC(=O)N2)c1ccc(Cl)cc1,0.652887,None
